In [1]:
import os
import pandas as pd
from maomao.utils.constants import *
from maomao.parsing.parsing_utils import *
from maomao.parsing.integrated_dataset_utils import *

#### Processing and standardizing peptide datasets (Hemolytik 2.0)

This notebook curates the **Hemolytik 2.0** dataset, which contains exclusively hemolytic peptides. The source provides both natural sequences and chemically modified variants. Here we parse and standardize both groups, apply duplicate consistency checks independently, and export curated datasets and metadata for downstream analysis.

- **Toxic effect / endpoint:** hemolytic
- **Source:** hemolytik 2.0
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Loads natural hemolytic peptides** from FASTA files whose filenames contain `"natural"`:
  - parses sequences using a standard FASTA reader,
  - assigns a positive label (`label = 1`) to all entries.
- **Loads modified hemolytic peptides** from `modified.fasta`:
  - uses a robust FASTA parser that tolerates non-standard characters,
  - assigns a positive label (`label = 1`) to all entries.
- **Separates natural and modified datasets** and processes them independently.
- **Checks duplicated sequences** within each group:
  - unique sequences are retained,
  - duplicates with consistent labels are collapsed,
  - sequences with conflicting labels (if any) are flagged as errors.
- **Builds metadata** from the project-wide Excel description sheet and appends QC statistics.
- **Exports curated outputs**:
  - `processed_hemolytic_dataset.csv` (natural peptides),
  - `modified_hemolytic_dataset.csv` (modified peptides),
  - `metadata.json`.

In [2]:
name_source = "hemolytik 2.0"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
input_dir = Path(PATH_INPUT) / name_source
dfs = []

for file in input_dir.iterdir():
    if "natural" in file.name.lower():
        df = read_fasta_doc(file)
        df["label"] = 1
        df = df[["sequence", "label"]]
        dfs.append(df)

df_non_modified = pd.concat(dfs, ignore_index=True)
df_non_modified.shape

(9921, 2)

In [4]:
df_modified = (
    read_fasta_with_strange_character(f"{PATH_INPUT}/{name_source}/modified.fasta")
    .assign(label=1)
    [["sequence", "label"]]
)
df_modified

,sequence,label
0,GW*LR**K**AAK**SVGK**FY*Y*K**HK*Y*Y*IK*AAWQIGKHAL,1
1,GW*LR**K**AAK**SVGK**FY*Y*K**HK*Y*Y*IK*AAWQIGKHAL,1
2,APC-L-ACPC-K-ACPC-L-APC-L-ACPC-K-ACPC-L-APC-L-...,1
3,APC-L-ACPC-L-APC-L-ACPC-K-ACPC-L-ACPC-K-ACPC-L...,1
4,ACPC-L-APC-K-APC-L-ACPC-K-APC-L-ACPC-L-ACPC-L-...,1
...,...,...
2623,ZCRRLCYKQRCVTYCRGR,1
2624,IlGPVLGLVGSALGGLLKKI,1
2625,IlGPVLGMVGSALGGLLKKI,1
2626,GLLSALOALGKLL,1


In [5]:
df_modified["is_canon"] = df_modified["sequence"].apply(check_sequence) # detect if the sequence is canonical
df_canon_from_modified = df_modified[df_modified["is_canon"] == True][["sequence", "label"]] # canonical sequences within the modified file
df_modified = df_modified[df_modified["is_canon"] == False][["sequence", "label"]] # keep only real modified

In [6]:
# add the canonicals to the unmodified dataset

df_non_modified = pd.concat(
    [df_non_modified, df_canon_from_modified],
    ignore_index=True
)

- Checking duplicates

In [7]:
df_remove_duplicated_nonmodified, df_errors_nonmodified, df_unique_nonmodified = processing_duplicated(df_non_modified, group_seq="sequence", sort_key="label")
df_full_nonmodified = pd.concat([df_unique_nonmodified, df_remove_duplicated_nonmodified], axis=0)

In [8]:
df_errors_nonmodified.shape

(0, 1)

In [9]:
df_remove_duplicated_mod, df_errors_mod, df_unique_mod = processing_duplicated(df_modified, group_seq="sequence", sort_key="label")
df_full_mod = pd.concat([df_unique_mod, df_remove_duplicated_mod], axis=0)

In [10]:
df_errors_mod.shape

(0, 1)

- Working with metada

In [11]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [12]:
raw_total_sequences = len(df_non_modified) + len(df_modified)

In [13]:
dict_metadata.update({
    "number_of_raw_sequences": int(raw_total_sequences),
    "number_of_sequences_retained": len(df_full_nonmodified),
    "number_of_positive_sequences": int((df_full_nonmodified["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full_nonmodified["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors_nonmodified)),
    "number_of_modified_sequences" : len(df_full_mod),
    "number_of_erroneous_modified_sequences" : len(df_errors_mod),
    "modified_sequences_included": False,})

dict_metadata

{'type source': 'Database',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2025,
 'last update date': datetime.datetime(2014, 5, 14, 0, 0),
 'download date': Timestamp('2025-10-02 00:00:00'),
 'file format': 'fasta',
 'peptide property': 'hemolytic, toxic',
 'dataset information': 'Positive',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'No information',
 'repository or server': 'https://webs.iiitd.edu.in/raghava/hemolytik2/index.html',
 'publication': 'https://www.biorxiv.org/content/10.1101/2025.05.12.653624v1.full',
 'number_of_raw_sequences': 12549,
 'number_of_sequences_retained': 4960,
 'number_of_positive_sequences': 4960,
 'number_of_negative_sequences': 0,
 'number_of_erroneous_sequences': 0,
 'number_of_modified_sequences': 1380,
 'number_of_erroneous_modified_sequences': 0,
 'modified_sequences_included': False}

- Exporting data

In [14]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [15]:
df_full_nonmodified.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_hemolytic_dataset.csv", index=False)
df_full_mod.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/modified_hemolytic_dataset.csv", index=False)